# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment the next line if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and `@id`s.

In [ ]:
# List record sets and their fields
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Available record sets and their fields (@id):")
    for rs in record_sets:
        print(f' - Record set name: "{rs.name}", @id: "{rs.id}"')
        print("   Fields:")
        for f in rs.fields:
            print(f'      • {f.name} (@id: {f.id}) [dataType: {f.data_type}]')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Extract all record sets into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with {len(df)} records and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# For demonstration, use the first available record set for EDA
if record_set_ids:
    selected_record_set = record_set_ids[0]
    print(f"\nColumns in record set {selected_record_set}:")
    print(dataframes[selected_record_set].columns.tolist())
    dataframes[selected_record_set].head()
else:
    print('No record set data available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
import numpy as np

# Pick the first numeric field for demonstration
if record_set_ids:
    df = dataframes[selected_record_set]
    # Try to automatically pick first numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fiu' else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical column (if any exists)
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        group_field = None
        for col in categorical_cols:
            if col != numeric_field and df[col].nunique() < 30:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields found in the record set.")
else:
    print('No record set data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and len(df) > 0 and len(df.select_dtypes(include=[np.number]).columns) > 0:
    # Example: Histogram of the selected numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If there are at least 2 numeric fields, show scatter
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) >= 2:
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[num_cols[0]], y=df[num_cols[1]])
        plt.xlabel(num_cols[0])
        plt.ylabel(num_cols[1])
        plt.title(f'{num_cols[0]} vs. {num_cols[1]}')
        plt.show()
else:
    print("Not enough data to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- We explored the Mass Spectrometry dataset metadata and found available record sets and fields using their `@id`s.
- Data was extracted into dataframes using the `mlcroissant` API.
- A numeric field was selected for filtering and normalization, providing insights into its distribution.
- Basic visualizations (histogram, scatterplot) were produced where possible.

This workflow may be extended for advanced analytics, modeling, or domain-specific tasks.